In [0]:
# ==============================================================================
# GUARDA DE EXECUÇÃO — SÓ MOVE SE 1.1, 1.2, 1.3 E 1.4 ATUALIZARAM COM SUCESSO
# ==============================================================================
# Usa o histórico Delta (DESCRIBE HISTORY) das tabelas de saída de cada notebook:
#   • 1.1 material_cadastrao: carga (CTAS/WRITE) posterior aos arquivos em current/
#   • 1.2 material_inventory_history: escrita posterior à carga do 1.1 e com
#     linhas na reference_date do 1.1
#   • 1.3 material_spec_changes: MERGE/WRITE posterior à carga do 1.1
#   • 1.4 material_cadeia: carga (CTAS/WRITE) posterior à carga do 1.1
# Se algum notebook falhou (ou não rodou) depois da carga do 1.1, sua tabela fica
# com escrita antiga e a guarda levanta RuntimeError — os arquivos permanecem em
# current/ para reprocessamento.
# ==============================================================================

from pyspark.sql import functions as F

CATALOG_SCHEMA = "parts_hdbk_sandbox.pr_cadastrao"
CURRENT_PATH = "/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/"

# Operações Delta que representam escrita de dados (ignora ALTER/COMMENT/TAGS)
LOAD_OPS = ["CREATE TABLE AS SELECT", "CREATE OR REPLACE TABLE AS SELECT",
            "REPLACE TABLE AS SELECT", "WRITE"]
DATA_OPS = LOAD_OPS + ["MERGE", "UPDATE", "DELETE"]


def last_commit_ms(table, ops):
    """Epoch (ms) do último commit da tabela com operação em `ops`, ou None."""
    row = (
        spark.sql(f"DESCRIBE HISTORY {CATALOG_SCHEMA}.{table}")
        .filter(F.col("operation").isin(ops))
        .select(F.max(F.unix_millis("timestamp")).alias("ms"))
        .first()
    )
    return row.ms


def fmt(ms):
    return "nunca" if ms is None else spark.sql(
        f"SELECT CAST(timestamp_millis({ms}) AS STRING)").first()[0] + " UTC"


# ------------------------------------------------------------------------------
# Arquivos em current/
# ------------------------------------------------------------------------------
current_files = [f for f in dbutils.fs.ls(CURRENT_PATH) if not f.isDir()]
if not current_files:
    print("Nenhum arquivo em current/ — nada a mover.")
    dbutils.notebook.exit("NO_FILES")

newest_file_ms = max(f.modificationTime for f in current_files)
print(f"Arquivos em current/: {[f.name for f in current_files]}")
print(f"Arquivo mais recente: {fmt(newest_file_ms)}\n")

problems = []

# ------------------------------------------------------------------------------
# 1.1 — material_cadastrao
# ------------------------------------------------------------------------------
t11 = last_commit_ms("material_cadastrao", LOAD_OPS)
print(f"1.1 material_cadastrao          última carga: {fmt(t11)}")
if t11 is None or t11 < newest_file_ms:
    problems.append("1.1 não carregou os arquivos atuais de current/ (carga anterior aos arquivos).")
    ref_date = None
else:
    ref_dates = [r[0] for r in spark.sql(
        f"SELECT DISTINCT reference_date FROM {CATALOG_SCHEMA}.material_cadastrao").collect()]
    if len(ref_dates) != 1 or ref_dates[0] is None:
        problems.append(f"1.1 material_cadastrao deve ter exatamente uma reference_date (encontrado: {ref_dates}).")
        ref_date = None
    else:
        ref_date = ref_dates[0]
        print(f"    reference_date: {ref_date}")

# ------------------------------------------------------------------------------
# 1.2, 1.3, 1.4 — escrita posterior à carga do 1.1
# ------------------------------------------------------------------------------
DOWNSTREAM = [
    ("1.2", "material_inventory_history", DATA_OPS),
    ("1.3", "material_spec_changes", DATA_OPS),
    ("1.4", "material_cadeia", LOAD_OPS),
]

for nb, table, ops in DOWNSTREAM:
    ts = last_commit_ms(table, ops)
    print(f"{nb} {table:<30} última escrita: {fmt(ts)}")
    if t11 is not None and (ts is None or ts < t11):
        problems.append(f"{nb} {table} não foi atualizada após a carga do 1.1 (falhou ou não rodou).")

# 1.2 — snapshot do mês de referência precisa existir
if ref_date is not None:
    n_inv = spark.sql(
        f"SELECT COUNT(*) FROM {CATALOG_SCHEMA}.material_inventory_history "
        f"WHERE reference_date = DATE'{ref_date}'").first()[0]
    print(f"    1.2 linhas em reference_date {ref_date}: {n_inv:,}")
    if n_inv == 0:
        problems.append(f"1.2 material_inventory_history não possui snapshot para {ref_date}.")

# ------------------------------------------------------------------------------
# Decisão
# ------------------------------------------------------------------------------
if problems:
    raise RuntimeError(
        "Guarda de execução: movimentação para history/ BLOQUEADA. Arquivos mantidos em current/.\n  - "
        + "\n  - ".join(problems)
    )

print("\n✓ 1.1, 1.2, 1.3 e 1.4 atualizados com sucesso — liberado para mover arquivos.")

In [0]:
# ==============================================================================
# MOVER ARQUIVOS PROCESSADOS PARA HISTORY/
# ==============================================================================
# Move os arquivos Excel processados de current/ para history/ após o
# sucesso do SCD2. Valida que current/ ficou vazio ao final.
# Em caso de falha na movimentação, levanta RuntimeError para evitar
# reprocessamento na próxima execução.
# ==============================================================================

# ==============================================================================
# MOVER TODOS OS ARQUIVOS DE current/ PARA history/
# ==============================================================================

SOURCE_PATH = "/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/current/"
HISTORY_PATH = "/Volumes/parts_hdbk_sandbox/pr_cadastrao/sap_cadastraorefinado/history/"

moved = []
failed = []

# Lista todos os arquivos existentes em current/
files = dbutils.fs.ls(SOURCE_PATH)

if not files:
    print("Nenhum arquivo encontrado em current/")
else:

    for file in files:

        # Ignora diretórios caso existam
        if file.isDir():
            continue

        source = file.path
        destination = f"{HISTORY_PATH}{file.name}"

        try:
            dbutils.fs.mv(source, destination)
            moved.append(file.name)
            print(f"Movido: {file.name} → history/")

        except Exception as e:
            failed.append((file.name, str(e)))
            print(f"ERRO ao mover {file.name}: {e}")

print(f"\nResumo: {len(moved)} arquivo(s) movido(s), {len(failed)} erro(s)")

if failed:
    raise RuntimeError(
        f"Falha ao mover {len(failed)} arquivo(s): "
        + "; ".join(f"{name}: {err}" for name, err in failed)
    )

# ------------------------------------------------------------------------------
# Validação final
# ------------------------------------------------------------------------------

remaining = [
    f for f in dbutils.fs.ls(SOURCE_PATH)
    if not f.isDir()
]

if remaining:
    raise RuntimeError(
        f"Restaram {len(remaining)} arquivo(s) em current/: "
        + ", ".join(f.name for f in remaining)
    )

print("✓ Diretório current/ vazio após movimentação.")